### new addition

In [0]:
# Configures runtime parameters for manual, Databricks Job, or future Airflow execution.

import re
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "catalog",
    "training_lab_classic"
)

dbutils.widgets.text(
    "schema",
    "cold_chain"
)

dbutils.widgets.text(
    "volume",
    "volume1"
)

dbutils.widgets.text(
    "run_id",
    ""
)

dbutils.widgets.text(
    "orchestrator",
    "databricks_job"
)

dbutils.widgets.text(
    "airflow_dag_id",
    ""
)

dbutils.widgets.text(
    "airflow_run_id",
    ""
)

dbutils.widgets.text(
    "databricks_job_id",
    ""
)

dbutils.widgets.text(
    "databricks_run_id",
    ""
)

dbutils.widgets.dropdown(
    "simulate_failure",
    "false",
    [
        "false",
        "true"
    ]
)

CATALOG = dbutils.widgets.get(
    "catalog"
).strip()

SCHEMA = dbutils.widgets.get(
    "schema"
).strip()

VOLUME = dbutils.widgets.get(
    "volume"
).strip()

RUN_ID_INPUT = dbutils.widgets.get(
    "run_id"
).strip()

ORCHESTRATOR = dbutils.widgets.get(
    "orchestrator"
).strip()

AIRFLOW_DAG_ID = dbutils.widgets.get(
    "airflow_dag_id"
).strip()

AIRFLOW_RUN_ID = dbutils.widgets.get(
    "airflow_run_id"
).strip()

DATABRICKS_JOB_ID = dbutils.widgets.get(
    "databricks_job_id"
).strip()

DATABRICKS_RUN_ID = dbutils.widgets.get(
    "databricks_run_id"
).strip()

SIMULATE_FAILURE = (
    dbutils.widgets
    .get(
        "simulate_failure"
    )
    .strip()
    .lower()
    == "true"
)

# Job execution uses the shared Job parameter.
# Manual notebook execution gets a labelled fallback.

RUN_ID = (
    RUN_ID_INPUT
    if RUN_ID_INPUT
    else f"manual_{uuid.uuid4()}"
)

for parameter_name, parameter_value in {
    "catalog": CATALOG,
    "schema": SCHEMA,
    "volume": VOLUME
}.items():

    assert re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        parameter_value
    ), (
        f"Invalid {parameter_name}: "
        f"{parameter_value}"
    )

assert re.fullmatch(
    r"[A-Za-z0-9_.:-]+",
    RUN_ID
), (
    f"Invalid run_id: {RUN_ID}"
)

assert ORCHESTRATOR != "", (
    "orchestrator cannot be empty"
)

NAMESPACE = (
    f"{CATALOG}.{SCHEMA}"
)

BASE_PATH = (
    f"/Volumes/{CATALOG}/"
    f"{SCHEMA}/{VOLUME}"
)

print("==============================================")
print("OPERATIONAL READINESS CONFIGURATION")
print("==============================================")

print("Namespace         :", NAMESPACE)
print("Volume            :", BASE_PATH)
print("Run ID            :", RUN_ID)
print("Orchestrator      :", ORCHESTRATOR)
print("Airflow DAG ID    :", AIRFLOW_DAG_ID or None)
print("Airflow Run ID    :", AIRFLOW_RUN_ID or None)
print("Databricks Job ID :", DATABRICKS_JOB_ID or None)
print("Databricks Run ID :", DATABRICKS_RUN_ID or None)
print("Simulate failure  :", SIMULATE_FAILURE)

In [0]:
# Defines pipeline tables, verifies required outputs, and creates the operational audit Delta table.

AUDIT_TABLE = (
    f"{NAMESPACE}.pipeline_run_audit"
)

BRONZE_SENSOR_TABLE = (
    f"{NAMESPACE}.bronze_sensor_events"
)

SILVER_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_events"
)

QUARANTINE_SENSOR_TABLE = (
    f"{NAMESPACE}.quarantine_sensor_events"
)

ENRICHED_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_enriched"
)

SILVER_SHIPMENTS_TABLE = (
    f"{NAMESPACE}.silver_shipments"
)

TEMPERATURE_WINDOWS_TABLE = (
    f"{NAMESPACE}.shipment_temperature_windows"
)

SHIPMENT_HEALTH_TABLE = (
    f"{NAMESPACE}.shipment_health"
)

DELIVERY_PERFORMANCE_TABLE = (
    f"{NAMESPACE}.delivery_performance"
)

required_pipeline_tables = [
    BRONZE_SENSOR_TABLE,
    SILVER_SENSOR_TABLE,
    QUARANTINE_SENSOR_TABLE,
    ENRICHED_SENSOR_TABLE,
    SILVER_SHIPMENTS_TABLE,
    TEMPERATURE_WINDOWS_TABLE,
    SHIPMENT_HEALTH_TABLE,
    DELIVERY_PERFORMANCE_TABLE
]

missing_pipeline_tables = [
    table_name
    for table_name in required_pipeline_tables
    if not spark.catalog.tableExists(
        table_name
    )
]

assert len(missing_pipeline_tables) == 0, ("Missing required pipeline tables: "
    + ", ".join(
        missing_pipeline_tables
    )
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE}
    (
        run_id STRING,
        orchestrator STRING,
        airflow_dag_id STRING,
        airflow_run_id STRING,
        databricks_job_id STRING,
        databricks_run_id STRING,
        status STRING,
        started_at TIMESTAMP,
        completed_at TIMESTAMP,
        input_rows BIGINT,
        accepted_rows BIGINT,
        rejected_rows BIGINT,
        rejected_percent DOUBLE,
        temperature_exception_count BIGINT,
        gold_temperature_window_rows BIGINT,
        shipment_health_rows BIGINT,
        delivery_performance_rows BIGINT,
        error_message STRING
    )
    USING DELTA
    """
)

assert spark.catalog.tableExists(
    AUDIT_TABLE
), (
    f"Audit table does not exist: "
    f"{AUDIT_TABLE}"
)

print("==============================================")
print("OPERATIONAL TABLE CHECK")
print("==============================================")

for table_name in required_pipeline_tables:

    row_count = (
        spark.table(
            table_name
        ).count()
    )

    print(
        table_name,
        ":",
        row_count,
        "rows"
    )

print()
print(
    "Audit table:",
    AUDIT_TABLE
)

print(
    "Required operational tables are available."
)

audit_schema = (
    spark.table(
        AUDIT_TABLE
    ).schema
)

print(
    "Existing audit columns:",
    len(audit_schema.fields)
)

print(
    [
        field.name
        for field in audit_schema.fields
    ]
)

In [0]:
# Registers the execution as RUNNING while keeping the run identifier idempotent.

started_at = datetime.now(
    timezone.utc
).replace(
    tzinfo=None
)

audit_schema = (
    spark.table(
        AUDIT_TABLE
    ).schema
)

running_payload = {
    "run_id": RUN_ID,
    "orchestrator": ORCHESTRATOR,

    "airflow_dag_id": (
        AIRFLOW_DAG_ID
        if AIRFLOW_DAG_ID
        else None
    ),

    "airflow_run_id": (
        AIRFLOW_RUN_ID
        if AIRFLOW_RUN_ID
        else None
    ),

    "databricks_job_id": (
        DATABRICKS_JOB_ID
        if DATABRICKS_JOB_ID
        else None
    ),

    "databricks_run_id": (
        DATABRICKS_RUN_ID
        if DATABRICKS_RUN_ID
        else None
    ),

    "status": "RUNNING",
    "started_at": started_at,
    "completed_at": None,

    "input_rows": None,
    "accepted_rows": None,
    "rejected_rows": None,
    "rejected_percent": None,

    "temperature_exception_count": None,
    "gold_temperature_window_rows": None,
    "shipment_health_rows": None,
    "delivery_performance_rows": None,

    "error_message": None
}

audit_columns = [
    field.name
    for field in audit_schema.fields
]

unexpected_columns = (
    set(running_payload.keys())
    - set(audit_columns)
)

assert not unexpected_columns, (
    "RUNNING payload has unknown columns: "
    + ", ".join(
        sorted(unexpected_columns)
    )
)

aligned_running_payload = {
    column_name: running_payload.get(
        column_name,
        None
    )
    for column_name in audit_columns
}

running_df = spark.createDataFrame(
    [aligned_running_payload],
    schema=audit_schema
)

audit_delta = DeltaTable.forName(
    spark,
    AUDIT_TABLE
)

(
    audit_delta
    .alias("target")
    .merge(
        running_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

audit_run_count = (
    spark.table(
        AUDIT_TABLE
    )
    .filter(
        F.col("run_id") == RUN_ID
    )
    .count()
)

assert audit_run_count == 1, (
    f"Expected exactly one audit record "
    f"for {RUN_ID}, found "
    f"{audit_run_count}"
)

print("==============================================")
print("PIPELINE RUN REGISTERED")
print("==============================================")

print("Run ID :", RUN_ID)
print("Status : RUNNING")

In [0]:
# Defines a reusable schema-safe function that updates the current pipeline audit record.

def update_audit_record(
    status,
    completed_at,
    metrics=None,
    error_message=None
):
    metrics = (
        metrics
        if metrics is not None
        else {}
    )

    audit_schema = (
        spark.table(
            AUDIT_TABLE
        ).schema
    )

    audit_columns = [
        field.name
        for field in audit_schema.fields
    ]

    audit_payload = {
        "run_id": RUN_ID,
        "orchestrator": ORCHESTRATOR,

        "airflow_dag_id": (
            AIRFLOW_DAG_ID
            if AIRFLOW_DAG_ID
            else None
        ),

        "airflow_run_id": (
            AIRFLOW_RUN_ID
            if AIRFLOW_RUN_ID
            else None
        ),

        "databricks_job_id": (
            DATABRICKS_JOB_ID
            if DATABRICKS_JOB_ID
            else None
        ),

        "databricks_run_id": (
            DATABRICKS_RUN_ID
            if DATABRICKS_RUN_ID
            else None
        ),

        "status": status,
        "started_at": started_at,
        "completed_at": completed_at,

        "input_rows": metrics.get(
            "input_rows"
        ),

        "accepted_rows": metrics.get(
            "accepted_rows"
        ),

        "rejected_rows": metrics.get(
            "rejected_rows"
        ),

        "rejected_percent": metrics.get(
            "rejected_percent"
        ),

        "temperature_exception_count": (
            metrics.get(
                "temperature_exception_count"
            )
        ),

        "gold_temperature_window_rows": (
            metrics.get(
                "gold_temperature_window_rows"
            )
        ),

        "shipment_health_rows": (
            metrics.get(
                "shipment_health_rows"
            )
        ),

        "delivery_performance_rows": (
            metrics.get(
                "delivery_performance_rows"
            )
        ),

        "error_message": error_message
    }

    unexpected_columns = (
        set(audit_payload.keys())
        - set(audit_columns)
    )

    assert not unexpected_columns, (
        "Audit payload has unknown columns: "
        + ", ".join(
            sorted(unexpected_columns)
        )
    )

    aligned_audit_payload = {
        column_name: audit_payload.get(
            column_name,
            None
        )
        for column_name in audit_columns
    }

    audit_record_df = spark.createDataFrame(
        [aligned_audit_payload],
        schema=audit_schema
    )

    audit_delta = DeltaTable.forName(
        spark,
        AUDIT_TABLE
    )

    (
        audit_delta
        .alias("target")
        .merge(
            audit_record_df.alias("source"),
            "target.run_id = source.run_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        "Audit record updated:",
        RUN_ID,
        status
    )

In [0]:
# Loads pipeline outputs and calculates all metrics required for operational validation.

bronze_sensor = spark.table(
    BRONZE_SENSOR_TABLE
)

silver_sensor = spark.table(
    SILVER_SENSOR_TABLE
)

quarantine_sensor = spark.table(
    QUARANTINE_SENSOR_TABLE
)

enriched_sensor = spark.table(
    ENRICHED_SENSOR_TABLE
)

silver_shipments = spark.table(
    SILVER_SHIPMENTS_TABLE
)

temperature_windows = spark.table(
    TEMPERATURE_WINDOWS_TABLE
)

shipment_health = spark.table(
    SHIPMENT_HEALTH_TABLE
)

delivery_performance = spark.table(
    DELIVERY_PERFORMANCE_TABLE
)

input_rows = bronze_sensor.count()

accepted_rows = silver_sensor.count()

rejected_rows = quarantine_sensor.count()

enriched_sensor_rows = enriched_sensor.count()

silver_shipment_rows = silver_shipments.count()

temperature_exception_count = (
    enriched_sensor
    .filter(
        F.col(
            "is_temperature_exception"
        ) == True
    )
    .count()
)

gold_temperature_window_rows = (
    temperature_windows.count()
)

shipment_health_rows = (
    shipment_health.count()
)

delivery_performance_rows = (
    delivery_performance.count()
)

rejected_percent = (
    float(
        rejected_rows
        / input_rows
        * 100.0
    )
    if input_rows > 0
    else None
)

gold_metrics_row = (
    temperature_windows
    .agg(
        F.coalesce(
            F.sum(
                "accepted_reading_count"
            ),
            F.lit(0)
        )
        .cast("long")
        .alias(
            "represented_readings"
        ),
        F.coalesce(
            F.sum(
                "exception_count"
            ),
            F.lit(0)
        )
        .cast("long")
        .alias(
            "represented_exceptions"
        )
    )
    .first()
)

gold_represented_readings = (
    gold_metrics_row[
        "represented_readings"
    ]
)

gold_represented_exceptions = (
    gold_metrics_row[
        "represented_exceptions"
    ]
)

duplicate_silver_event_ids = (
    silver_sensor
    .groupBy(
        "event_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_health_shipments = (
    shipment_health
    .groupBy(
        "shipment_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_delivery_shipments = (
    delivery_performance
    .groupBy(
        "shipment_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_quarantine_reasons = (
    quarantine_sensor
    .filter(
        F.col(
            "rejection_reason"
        ).isNull()
        |
        (
            F.trim(
                F.col(
                    "rejection_reason"
                )
            ) == ""
        )
    )
    .count()
)

invalid_gold_exception_rates = (
    temperature_windows
    .filter(
        F.col(
            "exception_rate"
        ).isNull()
        |
        (
            F.col(
                "exception_rate"
            ) < 0
        )
        |
        (
            F.col(
                "exception_rate"
            ) > 100
        )
    )
    .count()
)

operational_metrics = {
    "input_rows": input_rows,
    "accepted_rows": accepted_rows,
    "rejected_rows": rejected_rows,
    "rejected_percent": rejected_percent,
    "temperature_exception_count": (
        temperature_exception_count
    ),
    "gold_temperature_window_rows": (
        gold_temperature_window_rows
    ),
    "shipment_health_rows": (
        shipment_health_rows
    ),
    "delivery_performance_rows": (
        delivery_performance_rows
    )
}

validation_metrics = {
    "enriched_sensor_rows": (
        enriched_sensor_rows
    ),
    "silver_shipment_rows": (
        silver_shipment_rows
    ),
    "gold_represented_readings": (
        gold_represented_readings
    ),
    "gold_represented_exceptions": (
        gold_represented_exceptions
    ),
    "duplicate_silver_event_ids": (
        duplicate_silver_event_ids
    ),
    "duplicate_health_shipments": (
        duplicate_health_shipments
    ),
    "duplicate_delivery_shipments": (
        duplicate_delivery_shipments
    ),
    "invalid_quarantine_reasons": (
        invalid_quarantine_reasons
    ),
    "invalid_gold_exception_rates": (
        invalid_gold_exception_rates
    )
}

print("==============================================")
print("COLLECTED OPERATIONAL METRICS")
print("==============================================")

print("Input rows                  :", input_rows)
print("Accepted rows               :", accepted_rows)
print("Rejected rows               :", rejected_rows)
print("Rejected percent            :", rejected_percent)
print("Enriched sensor rows        :", enriched_sensor_rows)

print(
    "Temperature exceptions      :",
    temperature_exception_count
)

print(
    "Gold represented readings   :",
    gold_represented_readings
)

print(
    "Gold represented exceptions :",
    gold_represented_exceptions
)

print(
    "Gold temperature windows    :",
    gold_temperature_window_rows
)

print(
    "Silver shipment rows        :",
    silver_shipment_rows
)

print(
    "Shipment health rows        :",
    shipment_health_rows
)

print(
    "Delivery performance rows   :",
    delivery_performance_rows
)

print(
    "Duplicate Silver event IDs  :",
    duplicate_silver_event_ids
)

print(
    "Duplicate health shipments  :",
    duplicate_health_shipments
)

print(
    "Duplicate delivery shipments:",
    duplicate_delivery_shipments
)

print(
    "Invalid quarantine reasons  :",
    invalid_quarantine_reasons
)

print(
    "Invalid Gold exception rates:",
    invalid_gold_exception_rates
)

In [0]:
# Evaluates every operational rule and collects validation failures without prematurely stopping the audit process.

validation_failures = []

if accepted_rows + rejected_rows != input_rows:
    validation_failures.append(
        (
            "Sensor reconciliation failed: "
            f"accepted={accepted_rows}, "
            f"rejected={rejected_rows}, "
            f"input={input_rows}"
        )
    )

if enriched_sensor_rows != accepted_rows:
    validation_failures.append(
        (
            "Enrichment reconciliation failed: "
            f"enriched={enriched_sensor_rows}, "
            f"accepted={accepted_rows}"
        )
    )

if gold_represented_readings != accepted_rows:
    validation_failures.append(
        (
            "Gold reading reconciliation failed: "
            f"Gold={gold_represented_readings}, "
            f"Silver={accepted_rows}"
        )
    )

if (
    gold_represented_exceptions
    != temperature_exception_count
):
    validation_failures.append(
        (
            "Gold exception reconciliation failed: "
            f"Gold={gold_represented_exceptions}, "
            f"Silver={temperature_exception_count}"
        )
    )

if shipment_health_rows != silver_shipment_rows:
    validation_failures.append(
        (
            "Shipment-health reconciliation failed: "
            f"health={shipment_health_rows}, "
            f"shipments={silver_shipment_rows}"
        )
    )

if (
    delivery_performance_rows
    != silver_shipment_rows
):
    validation_failures.append(
        (
            "Delivery-performance reconciliation failed: "
            f"performance={delivery_performance_rows}, "
            f"shipments={silver_shipment_rows}"
        )
    )

if duplicate_silver_event_ids != 0:
    validation_failures.append(
        (
            "Duplicate Silver event IDs found: "
            f"{duplicate_silver_event_ids}"
        )
    )

if duplicate_health_shipments != 0:
    validation_failures.append(
        (
            "Duplicate shipment-health rows found: "
            f"{duplicate_health_shipments}"
        )
    )

if duplicate_delivery_shipments != 0:
    validation_failures.append(
        (
            "Duplicate delivery-performance rows found: "
            f"{duplicate_delivery_shipments}"
        )
    )

if invalid_quarantine_reasons != 0:
    validation_failures.append(
        (
            "Quarantine rows without rejection reasons: "
            f"{invalid_quarantine_reasons}"
        )
    )

if invalid_gold_exception_rates != 0:
    validation_failures.append(
        (
            "Invalid Gold exception rates found: "
            f"{invalid_gold_exception_rates}"
        )
    )

if SIMULATE_FAILURE:
    validation_failures.append(
        "Intentional operational-readiness failure requested."
    )

validation_failure_count = len(
    validation_failures
)

print("==============================================")
print("OPERATIONAL VALIDATION RESULTS")
print("==============================================")

print(
    "Validation failures:",
    validation_failure_count
)

if validation_failure_count == 0:
    print(
        "All operational validation rules passed."
    )
else:
    for index, failure in enumerate(
        validation_failures,
        start=1
    ):
        print(
            f"{index}. {failure}"
        )



In [0]:
# Finalizes the audit record from the collected validation results and fails the task when required.

completed_at = datetime.now(
    timezone.utc
).replace(
    tzinfo=None
)

if validation_failure_count == 0:
    update_audit_record(
        status="SUCCESS",
        completed_at=completed_at,
        metrics=operational_metrics,
        error_message=None
    )

    final_status = "SUCCESS"
    final_error_message = None

    print("==============================================")
    print("OPERATIONAL READINESS RESULT")
    print("==============================================")

    print("Run ID :", RUN_ID)
    print("Status : SUCCESS")

    print()
    print("Input rows                  :", input_rows)
    print("Accepted rows               :", accepted_rows)
    print("Rejected rows               :", rejected_rows)
    print("Rejected percent            :", rejected_percent)
    print("Enriched sensor rows        :", enriched_sensor_rows)

    print(
        "Temperature exceptions      :",
        temperature_exception_count
    )

    print(
        "Gold represented readings   :",
        gold_represented_readings
    )

    print(
        "Gold represented exceptions :",
        gold_represented_exceptions
    )

    print(
        "Gold temperature windows    :",
        gold_temperature_window_rows
    )

    print(
        "Shipment health rows        :",
        shipment_health_rows
    )

    print(
        "Delivery performance rows   :",
        delivery_performance_rows
    )

    print()
    print(
        "Audit record finalized successfully."
    )

else:
    final_status = "FAILED"

    final_error_message = (
        " | ".join(
            validation_failures
        )[:4000]
    )

    update_audit_record(
        status="FAILED",
        completed_at=completed_at,
        metrics=operational_metrics,
        error_message=final_error_message
    )

    print("==============================================")
    print("OPERATIONAL READINESS RESULT")
    print("==============================================")

    print("Run ID :", RUN_ID)
    print("Status : FAILED")
    print("Error  :", final_error_message)

if final_status == "FAILED":
    raise RuntimeError(
        final_error_message
    )

In [0]:
# Verifies the persisted audit record against the current pipeline outputs.

current_run = (
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
)

current_run_count = current_run.count()

assert current_run_count == 1, (
    f"Expected exactly one audit record "
    f"for run_id={RUN_ID}, "
    f"found {current_run_count}"
)

run = current_run.first()

assert run is not None, (
    f"Audit record not found for run_id={RUN_ID}"
)

assert run["status"] == "SUCCESS", (
    f"Expected audit status SUCCESS, "
    f"found {run['status']}"
)

assert run["started_at"] is not None, (
    "Audit started_at is missing"
)

assert run["completed_at"] is not None, (
    "Audit completed_at is missing"
)

assert (
    run["completed_at"] >= run["started_at"]
), (
    "Audit completed_at is earlier than started_at"
)

assert run["input_rows"] is not None, (
    "Audit input_rows is missing"
)

assert run["accepted_rows"] is not None, (
    "Audit accepted_rows is missing"
)

assert run["rejected_rows"] is not None, (
    "Audit rejected_rows is missing"
)

assert (
    run["accepted_rows"]
    + run["rejected_rows"]
    == run["input_rows"]
), (
    f"Stored sensor reconciliation failed: "
    f"accepted={run['accepted_rows']}, "
    f"rejected={run['rejected_rows']}, "
    f"input={run['input_rows']}"
)

actual_input_rows = (
    spark.table(
        BRONZE_SENSOR_TABLE
    ).count()
)

actual_accepted_rows = (
    spark.table(
        SILVER_SENSOR_TABLE
    ).count()
)

actual_rejected_rows = (
    spark.table(
        QUARANTINE_SENSOR_TABLE
    ).count()
)

actual_temperature_exception_count = (
    spark.table(
        ENRICHED_SENSOR_TABLE
    )
    .filter(
        F.col(
            "is_temperature_exception"
        ) == True
    )
    .count()
)

actual_temperature_window_rows = (
    spark.table(
        TEMPERATURE_WINDOWS_TABLE
    ).count()
)

actual_shipment_health_rows = (
    spark.table(
        SHIPMENT_HEALTH_TABLE
    ).count()
)

actual_delivery_performance_rows = (
    spark.table(
        DELIVERY_PERFORMANCE_TABLE
    ).count()
)

assert (
    run["input_rows"]
    == actual_input_rows
), (
    f"Input-row audit mismatch: "
    f"audit={run['input_rows']}, "
    f"table={actual_input_rows}"
)

assert (
    run["accepted_rows"]
    == actual_accepted_rows
), (
    f"Accepted-row audit mismatch: "
    f"audit={run['accepted_rows']}, "
    f"table={actual_accepted_rows}"
)

assert (
    run["rejected_rows"]
    == actual_rejected_rows
), (
    f"Rejected-row audit mismatch: "
    f"audit={run['rejected_rows']}, "
    f"table={actual_rejected_rows}"
)

assert (
    run["temperature_exception_count"]
    == actual_temperature_exception_count
), (
    f"Temperature-exception audit mismatch: "
    f"audit={run['temperature_exception_count']}, "
    f"table={actual_temperature_exception_count}"
)

assert (
    run["gold_temperature_window_rows"]
    == actual_temperature_window_rows
), (
    f"Temperature-window audit mismatch: "
    f"audit={run['gold_temperature_window_rows']}, "
    f"table={actual_temperature_window_rows}"
)

assert (
    run["shipment_health_rows"]
    == actual_shipment_health_rows
), (
    f"Shipment-health audit mismatch: "
    f"audit={run['shipment_health_rows']}, "
    f"table={actual_shipment_health_rows}"
)

assert (
    run["delivery_performance_rows"]
    == actual_delivery_performance_rows
), (
    f"Delivery-performance audit mismatch: "
    f"audit={run['delivery_performance_rows']}, "
    f"table={actual_delivery_performance_rows}"
)

if run["input_rows"] > 0:
    expected_rejected_percent = (
        run["rejected_rows"]
        / run["input_rows"]
        * 100.0
    )

    assert (
        run["rejected_percent"] is not None
    ), (
        "rejected_percent is missing"
    )

    assert abs(
        run["rejected_percent"]
        - expected_rejected_percent
    ) < 0.000001, (
        f"Rejected-percent audit mismatch: "
        f"audit={run['rejected_percent']}, "
        f"expected={expected_rejected_percent}"
    )

else:
    assert (
        run["rejected_percent"] is None
    ), (
        "rejected_percent should be NULL "
        "when input_rows is zero"
    )

assert run["error_message"] is None, (
    f"Successful run contains an error message: "
    f"{run['error_message']}"
)

print("==============================================")
print("OPERATIONAL READINESS CERTIFICATE")
print("==============================================")

print()
print("EXECUTION")
print("----------------------------------------------")

print(
    "Run ID             :",
    run["run_id"]
)

print(
    "Status             :",
    run["status"]
)

print(
    "Orchestrator       :",
    run["orchestrator"]
)

print(
    "Airflow DAG ID     :",
    run["airflow_dag_id"]
)

print(
    "Airflow Run ID     :",
    run["airflow_run_id"]
)

print(
    "Databricks Job ID  :",
    run["databricks_job_id"]
)

print(
    "Databricks Run ID  :",
    run["databricks_run_id"]
)

print()
print("TIMING")
print("----------------------------------------------")

print(
    "Started at         :",
    run["started_at"]
)

print(
    "Completed at       :",
    run["completed_at"]
)

print()
print("PIPELINE COUNTS")
print("----------------------------------------------")

print(
    "Input rows         :",
    run["input_rows"]
)

print(
    "Accepted rows      :",
    run["accepted_rows"]
)

print(
    "Rejected rows      :",
    run["rejected_rows"]
)

print(
    "Rejected percent   :",
    run["rejected_percent"]
)

print()
print("PIPELINE QUALITY")
print("----------------------------------------------")

print(
    "Temperature exceptions :",
    run["temperature_exception_count"]
)

print(
    "Temperature windows    :",
    run["gold_temperature_window_rows"]
)

print(
    "Shipment health rows   :",
    run["shipment_health_rows"]
)

print(
    "Delivery rows          :",
    run["delivery_performance_rows"]
)

print()
print("==============================================")
print("OPERATIONAL READINESS STATUS: PASSED")
print("==============================================")

display(
    current_run
)

### new till here

In [0]:
import re
from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "catalog",
    "training_lab_classic"
)

dbutils.widgets.text(
    "schema",
    "cold_chain"
)

dbutils.widgets.text(
    "volume",
    "volume1"
)

dbutils.widgets.text(
    "run_id",
    "manual_001"
)

dbutils.widgets.text(
    "as_of_time",
    "2026-10-09T17:00:00Z"
)

dbutils.widgets.text(
    "orchestrator",
    "manual"
)

dbutils.widgets.text(
    "airflow_dag_id",
    ""
)

dbutils.widgets.text(
    "airflow_run_id",
    ""
)

dbutils.widgets.dropdown(
    "simulate_failure",
    "false",
    ["false", "true"]
)

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

RUN_ID = dbutils.widgets.get("run_id")
AS_OF_TIME = dbutils.widgets.get("as_of_time")

ORCHESTRATOR = dbutils.widgets.get(
    "orchestrator"
)

AIRFLOW_DAG_ID = dbutils.widgets.get(
    "airflow_dag_id"
)

AIRFLOW_RUN_ID = dbutils.widgets.get(
    "airflow_run_id"
)

SIMULATE_FAILURE = (
    dbutils.widgets
    .get("simulate_failure")
    .lower()
    == "true"
)

for name, value in {
    "catalog": CATALOG,
    "schema": SCHEMA,
    "volume": VOLUME
}.items():
    assert re.fullmatch(
        r"[A-Za-z_][A-Za-z0-9_]*",
        value
    ), f"Invalid {name}: {value}"

assert re.fullmatch(
    r"[A-Za-z0-9_.:-]+",
    RUN_ID
), f"Invalid run_id: {RUN_ID}"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

BASE_PATH = (
    f"/Volumes/{CATALOG}/"
    f"{SCHEMA}/{VOLUME}"
)

print("==============================================")
print("COLD CHAIN PIPELINE CONFIGURATION")
print("==============================================")

print("Namespace        :", NAMESPACE)
print("Volume path      :", BASE_PATH)
print("Run ID           :", RUN_ID)
print("Evaluation time  :", AS_OF_TIME)
print("Orchestrator     :", ORCHESTRATOR)
print("Airflow DAG ID   :", AIRFLOW_DAG_ID)
print("Airflow Run ID   :", AIRFLOW_RUN_ID)
print("Simulate failure :", SIMULATE_FAILURE)

In [0]:
AUDIT_TABLE = (
    f"{NAMESPACE}.pipeline_run_audit"
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE}
    (
        run_id STRING,
        orchestrator STRING,
        airflow_dag_id STRING,
        airflow_run_id STRING,
        databricks_job_id STRING,
        databricks_run_id STRING,
        status STRING,
        started_at TIMESTAMP,
        completed_at TIMESTAMP,
        input_rows BIGINT,
        accepted_rows BIGINT,
        rejected_rows BIGINT,
        rejected_percent DOUBLE,
        temperature_exception_count BIGINT,
        gold_temperature_window_rows BIGINT,
        shipment_health_rows BIGINT,
        delivery_performance_rows BIGINT,
        error_message STRING
    )
    USING DELTA
    """
)

print(
    "Audit table created/verified:",
    AUDIT_TABLE
)

In [0]:
assert spark.catalog.tableExists(
    AUDIT_TABLE
), "pipeline_run_audit was not created"

print("Audit table exists.")

spark.table(
    AUDIT_TABLE
).printSchema()

In [0]:
# Registers the current pipeline execution as RUNNING
# using the existing audit-table schema.

from datetime import datetime, timezone
from delta.tables import DeltaTable


started_at = datetime.now(
    timezone.utc
).replace(
    tzinfo=None
)


# Reads the actual current schema of the audit table.
# The table currently has 25 columns.

audit_schema = (
    spark.table(
        AUDIT_TABLE
    ).schema
)

audit_columns = [
    field.name
    for field in audit_schema.fields
]


# Named payload containing the values available
# when the operational-readiness run starts.

start_payload = {
    "run_id": RUN_ID,
    "orchestrator": ORCHESTRATOR,

    "airflow_dag_id": (
        AIRFLOW_DAG_ID
        if AIRFLOW_DAG_ID
        else None
    ),

    "airflow_run_id": (
        AIRFLOW_RUN_ID
        if AIRFLOW_RUN_ID
        else None
    ),

    "databricks_job_id": None,
    "databricks_run_id": None,

    "status": "RUNNING",
    "started_at": started_at,
    "completed_at": None,

    "input_rows": None,
    "accepted_rows": None,
    "rejected_rows": None,
    "rejected_percent": None,

    "temperature_exception_count": None,
    "gold_temperature_window_rows": None,
    "shipment_health_rows": None,
    "delivery_performance_rows": None,

    "error_message": None
}


# Fails clearly if code contains a field that does not
# exist in the physical audit table.

unexpected_start_columns = (
    set(start_payload.keys())
    - set(audit_columns)
)

assert not unexpected_start_columns, (
    "Start payload contains columns that do not "
    "exist in the audit table: "
    + ", ".join(
        sorted(
            unexpected_start_columns
        )
    )
)


# Aligns the payload with all 25 audit-table columns.
# Additional nullable columns receive None.

aligned_start_payload = {
    column_name: start_payload.get(
        column_name,
        None
    )
    for column_name in audit_columns
}


# Dictionary input maps values by column name rather
# than depending on Row position or field count.

start_df = spark.createDataFrame(
    [aligned_start_payload],
    schema=audit_schema
)


print(
    "Audit table columns:",
    len(audit_columns)
)

print(
    "Populated start fields:",
    len(start_payload)
)

print(
    "Start DataFrame columns:",
    len(start_df.columns)
)


assert len(
    start_df.columns
) == len(
    audit_columns
), (
    "Start DataFrame is not aligned with "
    "the audit table schema"
)


audit_delta = DeltaTable.forName(
    spark,
    AUDIT_TABLE
)


(
    audit_delta
    .alias("target")
    .merge(
        start_df.alias("source"),
        (
            "target.run_id "
            "= source.run_id"
        )
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


run_count = (
    spark.table(
        AUDIT_TABLE
    )
    .filter(
        F.col("run_id") == RUN_ID
    )
    .count()
)


assert run_count == 1, (
    f"Expected exactly one audit record "
    f"for {RUN_ID}, found {run_count}"
)


print(
    "Run registered:",
    RUN_ID
)

print(
    "Status:",
    "RUNNING"
)

In [0]:
from delta.tables import DeltaTable

audit_delta = DeltaTable.forName(
    spark,
    AUDIT_TABLE
)

(
    audit_delta.alias("target")
    .merge(
        start_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    "Run registered:",
    RUN_ID
)

In [0]:
display(
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
)

In [0]:
BRONZE_SENSOR_TABLE = (
    f"{NAMESPACE}.bronze_sensor_events"
)

SILVER_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_events"
)

QUARANTINE_SENSOR_TABLE = (
    f"{NAMESPACE}.quarantine_sensor_events"
)

ENRICHED_SENSOR_TABLE = (
    f"{NAMESPACE}.silver_sensor_enriched"
)

TEMPERATURE_WINDOWS_TABLE = (
    f"{NAMESPACE}.shipment_temperature_windows"
)

SHIPMENT_HEALTH_TABLE = (
    f"{NAMESPACE}.shipment_health"
)

DELIVERY_PERFORMANCE_TABLE = (
    f"{NAMESPACE}.delivery_performance"
)

required_tables = [
    BRONZE_SENSOR_TABLE,
    SILVER_SENSOR_TABLE,
    QUARANTINE_SENSOR_TABLE,
    ENRICHED_SENSOR_TABLE,
    TEMPERATURE_WINDOWS_TABLE,
    SHIPMENT_HEALTH_TABLE,
    DELIVERY_PERFORMANCE_TABLE
]

for table_name in required_tables:
    assert spark.catalog.tableExists(
        table_name
    ), f"Missing table: {table_name}"

input_rows = spark.table(
    BRONZE_SENSOR_TABLE
).count()

accepted_rows = spark.table(
    SILVER_SENSOR_TABLE
).count()

rejected_rows = spark.table(
    QUARANTINE_SENSOR_TABLE
).count()

temperature_exception_count = (
    spark.table(
        ENRICHED_SENSOR_TABLE
    )
    .filter(
        F.col(
            "is_temperature_exception"
        ) == True
    )
    .count()
)

gold_temperature_window_rows = (
    spark.table(
        TEMPERATURE_WINDOWS_TABLE
    )
    .count()
)

shipment_health_rows = (
    spark.table(
        SHIPMENT_HEALTH_TABLE
    )
    .count()
)

delivery_performance_rows = (
    spark.table(
        DELIVERY_PERFORMANCE_TABLE
    )
    .count()
)

if input_rows > 0:
    rejected_percent = (
        rejected_rows
        / input_rows
    ) * 100.0
else:
    rejected_percent = None


In [0]:
print("==============================================")
print("PIPELINE METRICS")
print("==============================================")

print("Input rows                  :", input_rows)
print("Accepted rows               :", accepted_rows)
print("Rejected rows               :", rejected_rows)
print("Rejected percent            :", rejected_percent)

print(
    "Temperature exceptions      :",
    temperature_exception_count
)

print(
    "Gold temperature windows    :",
    gold_temperature_window_rows
)

print(
    "Shipment health rows        :",
    shipment_health_rows
)

print(
    "Delivery performance rows   :",
    delivery_performance_rows
)

In [0]:
completed_at = datetime.now(
    timezone.utc
).replace(
    tzinfo=None
)


# Reads the current physical audit-table schema.
audit_schema = (
    spark.table(
        AUDIT_TABLE
    ).schema
)

audit_columns = [
    field.name
    for field in audit_schema.fields
]


# Creates the SUCCESS record using named fields.
success_payload = {
    "run_id": RUN_ID,
    "orchestrator": ORCHESTRATOR,

    "airflow_dag_id": (
        AIRFLOW_DAG_ID
        if AIRFLOW_DAG_ID
        else None
    ),

    "airflow_run_id": (
        AIRFLOW_RUN_ID
        if AIRFLOW_RUN_ID
        else None
    ),

    "databricks_job_id": None,
    "databricks_run_id": None,

    "status": "SUCCESS",
    "started_at": started_at,
    "completed_at": completed_at,

    "input_rows": int(
        input_rows
    ),

    "accepted_rows": int(
        accepted_rows
    ),

    "rejected_rows": int(
        rejected_rows
    ),

    "rejected_percent": (
        float(
            rejected_percent
        )
        if rejected_percent is not None
        else None
    ),

    "temperature_exception_count": int(
        temperature_exception_count
    ),

    "gold_temperature_window_rows": int(
        gold_temperature_window_rows
    ),

    "shipment_health_rows": int(
        shipment_health_rows
    ),

    "delivery_performance_rows": int(
        delivery_performance_rows
    ),

    "error_message": None
}


# Confirms that all supplied fields exist
# in the current audit-table schema.
unknown_success_columns = (
    set(success_payload.keys())
    - set(audit_columns)
)

assert not unknown_success_columns, (
    "Success payload contains unknown columns: "
    + ", ".join(
        sorted(
            unknown_success_columns
        )
    )
)


# Aligns the payload with all physical table columns.
# Additional nullable columns receive None.
aligned_success_payload = {
    column_name: success_payload.get(
        column_name,
        None
    )
    for column_name in audit_columns
}


success_df = spark.createDataFrame(
    [aligned_success_payload],
    schema=audit_schema
)


print(
    "Audit table column count:",
    len(audit_columns)
)

print(
    "Success DataFrame column count:",
    len(success_df.columns)
)

print(
    "Columns initialized as null:",
    [
        column_name
        for column_name in audit_columns
        if column_name not in success_payload
    ]
)


assert len(
    success_df.columns
) == len(
    audit_columns
), (
    "Success DataFrame is not aligned "
    "with the audit-table schema"
)

In [0]:
(
    audit_delta.alias("target")
    .merge(
        success_df.alias("source"),
        "target.run_id = source.run_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    "Audit record completed:",
    RUN_ID
)

In [0]:
display(
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
)

In [0]:
if SIMULATE_FAILURE:
    raise RuntimeError(
        "Intentional pipeline failure requested "
        "through simulate_failure=true"
    )

print(
    "Failure simulation disabled. "
    "Pipeline may proceed."
)

In [0]:
run_count = (
    spark.table(AUDIT_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
    .count()
)

print(
    "Audit records for",
    RUN_ID,
    ":",
    run_count
)

assert run_count == 1, (
    f"Expected one audit record for {RUN_ID}, "
    f"found {run_count}"
)

print(
    "Audit idempotency check: PASS"
)

In [0]:
# Phase 10 operational readiness: validates the audit record against the actual tables.
# No hardcoded row counts, so it works for any input volume (20, 10k, 50k...).

audit = spark.table(AUDIT_TABLE)

current_run = audit.filter(F.col("run_id") == RUN_ID)

print("==============================================")
print("PHASE 10 OPERATIONAL READINESS")
print("==============================================")

run_count = current_run.count()

assert run_count == 1, (
    f"Expected exactly one audit record for {RUN_ID}, found {run_count}"
)

run = current_run.first()

assert run["status"] == "SUCCESS", (
    f"Run status is {run['status']}, expected SUCCESS"
)

assert run["started_at"] is not None, "started_at is missing"
assert run["completed_at"] is not None, "completed_at is missing"

# Actual counts from the pipeline tables (same schema as the audit table)
NS_AUDIT = AUDIT_TABLE.rsplit(".", 1)[0]

actual_input    = spark.table(f"{NS_AUDIT}.bronze_sensor_events").count()
actual_accepted = spark.table(f"{NS_AUDIT}.silver_sensor_events").count()
actual_rejected = spark.table(f"{NS_AUDIT}.quarantine_sensor_events").count()

duplicates_removed = (
    actual_input - actual_accepted - actual_rejected
)

# Audit must match the real tables
assert run["input_rows"] == actual_input, (
    f"Audit input_rows {run['input_rows']} != Bronze {actual_input}"
)
assert run["accepted_rows"] == actual_accepted, (
    f"Audit accepted_rows {run['accepted_rows']} != Silver {actual_accepted}"
)
assert run["rejected_rows"] == actual_rejected, (
    f"Audit rejected_rows {run['rejected_rows']} != Quarantine {actual_rejected}"
)

# Sanity rules
assert actual_input > 0, "Bronze is empty"
assert actual_accepted > 0, "Silver is empty"
assert duplicates_removed >= 0, (
    "Silver + Quarantine is larger than Bronze; rows were duplicated"
)
assert run["temperature_exception_count"] is not None, (
    "temperature_exception_count is missing"
)
assert run["temperature_exception_count"] <= actual_accepted, (
    "More temperature exceptions than accepted rows"
)

assert run["gold_temperature_window_rows"] == spark.table(TEMPERATURE_WINDOWS_TABLE).count()
assert run["shipment_health_rows"] == spark.table(SHIPMENT_HEALTH_TABLE).count()
assert run["delivery_performance_rows"] == spark.table(DELIVERY_PERFORMANCE_TABLE).count()

print("Run ID                  :", run["run_id"])
print("Status                  :", run["status"])
print("Orchestrator            :", run["orchestrator"])
print("Input rows              :", run["input_rows"])
print("Accepted rows           :", run["accepted_rows"])
print("Rejected rows           :", run["rejected_rows"])
print("Duplicates removed      :", duplicates_removed)
print("Temperature exceptions  :", run["temperature_exception_count"])
print("Temperature windows     :", run["gold_temperature_window_rows"])
print("Shipment health rows    :", run["shipment_health_rows"])
print("Delivery rows           :", run["delivery_performance_rows"])

print()
print("==============================================")
print("PHASE 10 STATUS: PASS")
print("==============================================")